# Guided Beam Search — Beam Tree Visualisation & Examples

Shows the beam search process **step-by-step** for a single RAGTruth Summary example.
For each decoding step the table shows every candidate token with its
`logP`, `LL` (Lookback Lens groundedness score), `FS` (FActScore sentence score),
and composite `rerank` score.

- **Green rows** = kept by the reranker
- **Pink rows** = pruned
- **⚡** = LL/FS changed the selected token vs pure logP ranking
- **🔸** = sentence boundary: FActScore fired here

Also loads experiment results from `results_pilot/` once the run finishes.

In [ ]:
import os, sys, warnings
from pathlib import Path

assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN env var before running'
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.benchmark import load_ragtruth
from src.guided_beam_search import guided_beam_search
from src.factscore_turbo import FActScoreTurbo
from experiments.guided_generation.lib import load_generator, load_lookback_classifier, build_prompt

print('Imports OK')

In [2]:
# Load model once (~5 s on MPS)
model, tokenizer = load_generator('Qwen/Qwen2.5-1.5B-Instruct', device='mps', dtype='bfloat16')
classifier = load_lookback_classifier(
    PROJECT_ROOT / 'experiments/lookback_lens_qwen/results/classifier.pkl'
)
fs_scorer = FActScoreTurbo(model='qwen2.5:7b', temperature=0.0, max_facts=10, batch_verify=True)
print(f'Layers={model.config.num_hidden_layers}, heads={model.config.num_attention_heads}')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Layers=28, heads=12


In [3]:
# Pick a hallucinated Summary example
df = load_ragtruth(n_samples=60, task_filter='Summary', seed=42)
row = df[df['is_hallucinated']].iloc[0]

print(f'Hallucinated: {row["is_hallucinated"]}  |  Task: {row["task_type"]}')
print()
print('CONTEXT (first 600 chars):')
print(row['context'][:600])
print()
print('REFERENCE RESPONSE (hallucinated):')
print(row['response'])

Repo card metadata block was not found. Setting CardData to empty.


Hallucinated: True  |  Task: Summary

CONTEXT (first 600 chars):
Mercedes driver and F1 championship leader Lewis Hamilton stole pole position for Sunday's Chinese Grand Prix from teammate and fierce rival Nico Rosberg in dramatic fashion. Hamilton took first place on the front row on the last lap, beating Rosberg by a slim four hundredths of a second margin. Frenemies. The two former friends have enjoyed, or rather endured, a heated rivalry since falling out last season and Rosberg's annoyance at Hamilton's last ditch success was obvious. The German appeared upset as he left his car and refused to shake Hamilton's hand. He did, however, find time to congra

REFERENCE RESPONSE (hallucinated):
Lewis Hamilton stole the pole position from his Mercedes teammate Nico Rosberg in the final moments of qualifying for the Chinese Grand Prix, with a margin of just four hundredths of a second. Despite their heated rivalry, Hamilton remained magnanimous in defeat, focusing on his ultimate goal of w

In [4]:
# Run three conditions with full step logging
# Takes ~5-15 min (combined calls Ollama at sentence boundaries)
BEAM_WIDTH = 4
MAX_TOKENS = 64
SEED = 42

prompt = build_prompt(tokenizer, row['context'], row.get('question', ''))

print('Running vanilla ...')
res_vanilla = guided_beam_search(
    model, tokenizer, prompt, row['context'],
    lambda_ll=0.0, lambda_fs=0.0,
    beam_width=BEAM_WIDTH, max_new_tokens=MAX_TOKENS,
    log_per_step=True, seed=SEED,
)
print(f'  → {repr(res_vanilla.text[:120])}')
print(f'  reorders: {res_vanilla.n_ll_reorders}/{res_vanilla.n_total_steps}')

print('Running lookback only ...')
res_lookback = guided_beam_search(
    model, tokenizer, prompt, row['context'],
    lookback_classifier=classifier,
    lambda_ll=2.0, lambda_fs=0.0,
    score_mode='blend', blend_w=0.5, per_candidate_ll=False,
    beam_width=BEAM_WIDTH, max_new_tokens=MAX_TOKENS,
    log_per_step=True, seed=SEED,
)
print(f'  → {repr(res_lookback.text[:120])}')
print(f'  reorders: {res_lookback.n_ll_reorders}/{res_lookback.n_total_steps}')

print('Running combined (LL+FS) ...')
res_combined = guided_beam_search(
    model, tokenizer, prompt, row['context'],
    lookback_classifier=classifier,
    factscore_scorer=fs_scorer,
    lambda_ll=2.0, lambda_fs=1.0,
    score_mode='blend', blend_w=0.5, per_candidate_ll=False,
    beam_width=BEAM_WIDTH, max_new_tokens=MAX_TOKENS,
    log_per_step=True, seed=SEED,
)
print(f'  → {repr(res_combined.text[:120])}')
print(f'  reorders: {res_combined.n_ll_reorders}/{res_combined.n_total_steps}')
print(f'  FS boundaries: {res_combined.n_sentence_end_steps}')

Running vanilla ...
  → 'Lewis Hamilton stole pole position from Nico Rosberg in dramatic fashion at the Chinese Grand Prix.'
  reorders: 0/64
Running lookback only ...
  → 'Mercedes driver Lewis Hamilton won Chinese Grand Prix pole position by four-hundredths over Nico Rosberg, despite Rosber'
  reorders: 57/64
Running combined (LL+FS) ...
  → 'Mercedes driver Lewis Hamilton won Chinese Grand Prix pole position by four-hundredths over Nico Rosberg, despite Rosber'
  reorders: 57/64
  FS boundaries: 11


In [5]:
import pandas as pd
from IPython.display import HTML, display

def _e(s):
    return s.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')


def beam_tree_html(result, title: str, show_all: bool = False) -> str:
    """
    Render step_log as a colour-coded HTML table.

    Shows only interesting steps by default:
      - first 5 (warm-up)
      - steps where LL/FS changed the top token vs logP (⚡)
      - sentence-boundary steps (🔸)
    """
    if not result.step_log:
        return f'<p><b>{title}</b>: no step log (pass log_per_step=True)</p>'

    rows = [dict(
        step=r.step, parent=r.parent_beam_id,
        token=r.token_text, logp=r.token_logp,
        ll=r.ll_score, fs=r.fs_score, score=r.rerank_score,
        kept=r.kept, sent=r.sentence_end,
    ) for r in result.step_log]
    df = pd.DataFrame(rows)

    flip_steps, sent_steps = set(), set()
    for step, g in df.groupby('step'):
        top_rerank = g[g['kept']].sort_values('score', ascending=False)['token'].iloc[0]
        top_logp   = g.sort_values('logp', ascending=False)['token'].iloc[0]
        if top_rerank != top_logp:
            flip_steps.add(step)
        if g['sent'].any():
            sent_steps.add(step)

    interesting = flip_steps | sent_steps | set(range(min(5, result.n_total_steps)))

    html = '''
    <style>
      .bst{border-collapse:collapse;font-family:monospace;font-size:11px;width:100%}
      .bst th{background:#2c3e50;color:#fff;padding:4px 8px;text-align:left}
      .bst td{padding:3px 8px;border-bottom:1px solid #e0e0e0;white-space:pre}
      .kpt{background:#d4edda} .prn{background:#f8d7da}
      .snt{border-bottom:3px solid #e67e22!important}
      .flp{font-weight:bold}
      .sep td{background:#f0f0f0;font-style:italic;color:#888;font-family:sans-serif}
    </style>
    '''
    html += f'<h3>{_e(title)}</h3>'
    html += (f'<p style="font-size:12px">Steps: {result.n_total_steps} | '
             f'<b>LL reorders: {result.n_ll_reorders}</b> | '
             f'FS boundaries: {result.n_sentence_end_steps}</p>')
    html += f'<p><b>Output:</b> {_e(result.text)}</p>'
    if not show_all:
        hidden = result.n_total_steps - len(interesting)
        html += (f'<p style="font-size:11px;color:#555">Showing {len(interesting)}/{result.n_total_steps} steps '
                 f'(first 5 + ⚡ reorders + 🔸 FS; {hidden} uninteresting hidden).</p>')

    html += ('<table class="bst"><tr>'
             '<th>Step</th><th>Token</th><th>logP</th>'
             '<th>LL</th><th>FS</th><th>Score</th><th></th></tr>')

    prev = None
    for step, grp in df.groupby('step'):
        if not show_all and step not in interesting:
            continue
        if prev is not None and step > prev + 1:
            n = step - prev - 1
            html += f'<tr class="sep"><td colspan="7">··· {n} uninteresting step(s) ···</td></tr>'
        prev = step

        is_flip = step in flip_steps
        is_sent = step in sent_steps
        for _, r in grp.sort_values('score', ascending=False).iterrows():
            cls = 'kpt' if r['kept'] else 'prn'
            if is_sent: cls += ' snt'
            if is_flip and r['kept']: cls += ' flp'
            mark = ('⚡' if is_flip and r['kept'] else '') + ('🔸' if r['sent'] else '')
            html += (f'<tr class="{cls}">'
                     f'<td>{step}</td>'
                     f'<td>{_e(repr(r["token"]))}</td>'
                     f'<td>{r["logp"]:+.2f}</td>'
                     f'<td>{r["ll"]:+.3f}</td>'
                     f'<td>{r["fs"]:+.3f}</td>'
                     f'<td>{r["score"]:+.3f}</td>'
                     f'<td>{mark}</td></tr>')
    html += '</table>'
    return html


print('Visualisation helper ready.')

Visualisation helper ready.


In [6]:
display(HTML(beam_tree_html(res_vanilla, 'Vanilla (logP only)')))

Step,Token,logP,LL,FS,Score,
0,'Lewis',-0.25,+0.000,+0.500,-0.246,
0,'Mer',-2.12,+0.000,+0.500,-2.125,
0,'Hamilton',-3.12,+0.000,+0.500,-3.125,
0,'In',-4.12,+0.000,+0.500,-4.125,
1,' Hamilton',-0.00,+0.000,+0.500,-0.152,
1,'cedes',-0.00,+0.000,+0.500,-1.308,
1,' steals',-1.28,+0.000,+0.500,-2.712,
1,' stole',-1.53,+0.000,+0.500,-2.866,
1,' the',-1.01,+0.000,+0.500,-3.160,
1,' wins',-2.41,+0.000,+0.500,-3.405,


In [7]:
display(HTML(beam_tree_html(res_lookback, 'Lookback Lens only  (λ_LL=2.0)')))

Step,Token,logP,LL,FS,Score,
0,'Lewis',-0.25,+0.000,+0.500,+0.753,
0,'Mer',-2.12,+0.000,+0.500,+0.098,
0,'Hamilton',-3.12,+0.000,+0.500,-0.251,
0,'In',-4.12,+0.000,+0.500,-0.600,
1,' Hamilton',-0.00,+0.994,+0.500,+1.137,
1,'cedes',-0.00,+0.999,+0.500,+1.041,
1,' steals',-1.28,+0.994,+0.500,+0.618,
1,' stole',-1.53,+0.994,+0.500,+0.587,
1,' wins',-2.41,+0.994,+0.500,+0.477,
1,' won',-2.53,+0.994,+0.500,+0.461,


In [8]:
display(HTML(beam_tree_html(res_combined, 'Combined  (λ_LL=2.0, λ_FS=1.0)')))

Step,Token,logP,LL,FS,Score,
0,'Lewis',-0.25,+0.000,+0.500,+0.753,
0,'Mer',-2.12,+0.000,+0.500,+0.098,
0,'Hamilton',-3.12,+0.000,+0.500,-0.251,
0,'In',-4.12,+0.000,+0.500,-0.600,
1,' Hamilton',-0.00,+0.994,+0.500,+1.137,
1,'cedes',-0.00,+0.999,+0.500,+1.041,
1,' steals',-1.28,+0.994,+0.500,+0.618,
1,' stole',-1.53,+0.994,+0.500,+0.587,
1,' wins',-2.41,+0.994,+0.500,+0.477,
1,' won',-2.53,+0.994,+0.500,+0.461,


In [9]:
# Side-by-side output comparison
COLS = {'vanilla': ('#e8f5e9', res_vanilla),
        'lookback': ('#e3f2fd', res_lookback),
        'combined': ('#fff3e0', res_combined)}

html = '<h3>Side-by-side outputs</h3>'
html += f'<div style="background:#f5f5f5;padding:8px;font-size:11px;font-family:monospace;margin-bottom:12px">'
html += f'<b>Context (first 400 chars):</b><br>{_e(row["context"][:400])}…</div>'
html += '<table style="width:100%;border-collapse:collapse"><tr>'
for name, (bg, res) in COLS.items():
    html += (f'<td style="width:33%;vertical-align:top;padding:8px;'
             f'border:1px solid #ccc;background:{bg}">'
             f'<b>{name}</b><hr style="margin:4px 0"/>'
             f'<div style="font-size:12px">{_e(res.text)}</div>'
             f'<div style="font-size:10px;color:#555;margin-top:6px">'
             f'reorders: {res.n_ll_reorders}/{res.n_total_steps} | '
             f'FS fires: {res.n_sentence_end_steps}</div></td>')
html += '</tr></table>'
display(HTML(html))

vanillaLewis Hamilton stole pole position from Nico Rosberg in dramatic fashion at the Chinese Grand Prix.reorders: 0/64 | FS fires: 12,"lookbackMercedes driver Lewis Hamilton won Chinese Grand Prix pole position by four-hundredths over Nico Rosberg, despite Rosberg's frustration. Rosberg refused to shake Hamilton's hand. Mercedes teammate Kimi Raikkonen starts second. Hamilton enjoys support in Shanghai, where he thrives. Rosberg admits he was frustrated butreorders: 57/64 | FS fires: 12","combinedMercedes driver Lewis Hamilton won Chinese Grand Prix pole position by four-hundredths over Nico Rosberg, despite Rosberg's frustration. Rosberg refused to shake Hamilton's hand. Mercedes teammate Kimi Raikkonen starts second. Hamilton enjoys support in Shanghai, where he thrives. Rosberg admits he was frustrated.reorders: 57/64 | FS fires: 11"


In [10]:
# Show full beam tree (all steps) — uncheck to inspect every step
# display(HTML(beam_tree_html(res_combined, 'Combined — ALL STEPS', show_all=True)))

## Pilot experiment results

Run the cell below **after** the pilot experiment finishes (`/tmp/pilot_v2_done.flag` exists).

In [11]:
import json, pandas as pd
RESULTS = PROJECT_ROOT / 'experiments/guided_generation/results_pilot'

out_path   = RESULTS / 'outputs.parquet'
judge_path = RESULTS / 'judge_scores.csv'

if not out_path.exists():
    print('outputs.parquet not ready — re-run when experiment finishes')
else:
    df_out = pd.read_parquet(out_path)
    print(f'Outputs: {len(df_out)} rows  ({df_out["condition"].nunique()} conditions)')
    print(df_out.groupby('condition')[['n_response_tokens','elapsed_s']].mean().round(2))

if judge_path.exists() and judge_path.stat().st_size > 10:
    df_j = pd.read_csv(judge_path)
    print()
    print('Judge scores (mean per condition):')
    cols = ['faithfulness_mean','completeness_mean','coherence_mean']
    avail = [c for c in cols if c in df_j.columns]
    print(df_j.groupby('condition')[avail].mean().round(3))
else:
    print('judge_scores.csv not ready yet')

outputs.parquet not ready — re-run when experiment finishes
judge_scores.csv not ready yet
